In [1]:
%pwd

'a:\\projects\\text-summarizer\\research'

In [2]:
import os
os.chdir("../")

In [4]:
%pwd

'a:\\projects\\text-summarizer'

# Update entity

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

# Update configuration manager

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfiguartionManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories(path_to_directories=[self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories(path_to_directories=[config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

# Update components

In [8]:
import os
import urllib.request as request
import py7zr
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded succesfully with following info \n{headers}")
        else:
            logger.info(f"File already exists of size {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with py7zr.SevenZipFile(self.config.local_data_file, mode='r') as archieve:
            archieve.extractall(path=unzip_path)
        logger.info(f"Extracted archieve content to {unzip_path}")
        

# Update pipeline

In [10]:
try:
    config = ConfiguartionManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-06-18 11:34:29,115: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-18 11:34:29,119: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-18 11:34:29,121: INFO: common: Created directory at: artifacts]
[2026-06-18 11:34:29,123: INFO: common: Created directory at: artifacts/data_ingestion]
[2026-06-18 11:34:33,204: INFO: 3474932688: artifacts/data_ingestion/data.7z downloaded succesfully with following info 
Connection: close
Content-Length: 2944100
via: 1.1 google, 1.1 varnish, 1.1 varnish, 1.1 varnish
last-modified: Wed, 11 Aug 2021 04:21:50 GMT
server: Google Frontend
x-cloud-trace-context: b42714547d73dedf2588f7635a79badf
content-type: application/x-7z-compressed
etag: "sha256:57947d3dbdbba32300b85b33f0c88e59abdbeb4ba838246d26287c60b3a817ef"
Accept-Ranges: bytes
Age: 414830
Date: Thu, 18 Jun 2026 06:04:29 GMT
X-Served-By: cache-lga21984-LGA, cache-lga21984-LGA, cache-lga21926-LGA, cache-qaf-vabp4260023-QAF
X-Cache: MISS, HIT, HIT
X-Timer